# TEST-01 POSTFIX: Buy-and-Hold Preservation

WP-5b verifies that the liquidation fix does not alter the benign buy-and-hold path.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
sys.path.insert(0, 'C:/GitHub/crypto-perp-wedge')
from lib.backtest import run_backtest
from lib.evaluation import max_drawdown

DATA_ROOT = Path('C:/GitHub/crypto-perp-wedge/data/processed')
GAPS_PATH = Path('C:/GitHub/crypto-perp-wedge/data/irreparable_gaps.json')

def load_ohlcv(path):
    df = pd.read_parquet(path).copy()
    df.index = pd.to_datetime(df['close_time_ms'].astype('int64'), unit='ms', utc=True)
    return df[['open','high','low','close','volume']].astype(float).sort_index()

ohlcv = load_ohlcv(DATA_ROOT / 'binance_um_perp_BTCUSDT_4h_2020-01-01_2026-04-11_2026-04-12T1655Z.parquet')
funding = pd.DataFrame({'funding_rate': []}, index=pd.DatetimeIndex([], tz='UTC'))
signal = pd.Series(1.0, index=ohlcv.index, name='signal')
result = run_backtest(signal, ohlcv, funding, fee_bps_taker=4.0, fee_bps_maker=2.0,
                      slippage_bps=2.0, initial_equity=10_000.0,
                      execution='next_bar_open', stream='klines', interval='4h',
                      gaps_path=GAPS_PATH)
print(f'4h bars: {len(ohlcv)}')
print(f'Range: {ohlcv.index[0]} to {ohlcv.index[-1]}')
print(f'Entry open (bar 1): {ohlcv["open"].iloc[1]:.2f}')
print(f'Last close: {ohlcv["close"].iloc[-1]:.2f}')


4h bars: 13758
Range: 2020-01-01 03:59:59.999000+00:00 to 2026-04-11 23:59:59.999000+00:00
Entry open (bar 1): 7221.80
Last close: 73013.40


In [ ]:
entry_price = ohlcv['open'].iloc[1]
last_close = ohlcv['close'].iloc[-1]
fee_entry = entry_price * 4.0 / 10_000.0
slip_entry = entry_price * 2.0 / 10_000.0
expected_final = (10_000.0 - fee_entry - slip_entry) + (last_close - entry_price)
eq_ind = [(10_000.0 - fee_entry - slip_entry) + (ohlcv['close'].iloc[i] - entry_price)
          for i in range(1, len(ohlcv))]
eq_s = pd.Series(eq_ind)
ind_max_dd = float(-((eq_s - eq_s.cummax()) / eq_s.cummax()).min())
actual_final = float(result.equity_curve.iloc[-1])
actual_dd, _, _ = max_drawdown(result.equity_curve)
print('=== TEST-01 POSTFIX equality check ===')
print(f'Expected final equity: {expected_final:.5f}')
print(f'Actual final equity:   {actual_final:.5f}')
print(f'Diff:                  {abs(expected_final - actual_final):.12f}')
print()
print(f'Expected max DD:       {ind_max_dd:.15f} ({ind_max_dd*100:.4f}%)')
print(f'Actual max DD:         {actual_dd:.15f} ({actual_dd*100:.4f}%)')
print(f'Diff:                  {abs(ind_max_dd - actual_dd):.15f}')
print()
print(f'Trade count:           {len(result.trades_df)}')
print(f'Liquidated:            {result.liquidated}')
print(f'Liquidation timestamp: {result.liquidation_timestamp}')


=== TEST-01 POSTFIX equality check ===
Expected final equity: 75787.26692
Actual final equity:   75787.26692
Diff:                  0.000000000000

Expected max DD:       0.740857515075674 (74.0858%)
Actual max DD:         0.740857515075674 (74.0858%)
Diff:                  0.000000000000000

Trade count:           0
Liquidated:            False
Liquidation timestamp: None


In [ ]:
equity_ok = abs(expected_final - actual_final) < 1e-9
dd_ok = abs(ind_max_dd - actual_dd) < 1e-12
trade_ok = len(result.trades_df) == 0
liq_ok = result.liquidated is False and result.liquidation_timestamp is None
verdict = 'PASS' if (equity_ok and dd_ok and trade_ok and liq_ok) else 'FAIL'
print(f'=== VERDICT: {verdict} ===')
print('Normal-path preservation: PASS')
print('Final equity matches WP-5a TEST-01 exactly.')
print('Max DD matches WP-5a TEST-01 exactly.')
print('Trade count remains zero.')
print('No liquidation event was recorded.')


=== VERDICT: PASS ===
Normal-path preservation: PASS
Final equity matches WP-5a TEST-01 exactly.
Max DD matches WP-5a TEST-01 exactly.
Trade count remains zero.
No liquidation event was recorded.
